# Python practice 7: classes, dataclasses and protocols

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-07-classes-and-dataclasses.ipynb)

13 exercises on classes, special methods, dataclasses, enums, ABCs and protocols. Each exercise names the [reference entries](../learning/python-reference-07-classes-and-dataclasses.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 7 easy, 5 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Alternate constructor with classmethod](#exercise-1-alternate-constructor-with-classmethod) | Easy | [Attributes and methods](../learning/python-reference-07-classes-and-dataclasses.ipynb#attributes-and-methods) |
| 2 | [Validated property](#exercise-2-validated-property) | Easy | [Properties](../learning/python-reference-07-classes-and-dataclasses.ipynb#properties) |
| 3 | [Money value object](#exercise-3-money-value-object) | Medium | [Representation, equality, hashing and ordering](../learning/python-reference-07-classes-and-dataclasses.ipynb#representation-equality-hashing-and-ordering) |
| 4 | [Iterable container class](#exercise-4-iterable-container-class) | Medium | [Container and callable protocols](../learning/python-reference-07-classes-and-dataclasses.ipynb#container-and-callable-protocols) |
| 5 | [Comparable versions](#exercise-5-comparable-versions) | Medium | [Representation, equality, hashing and ordering](../learning/python-reference-07-classes-and-dataclasses.ipynb#representation-equality-hashing-and-ordering) |
| 6 | [Frozen dataclass for dedupe](#exercise-6-frozen-dataclass-for-dedupe) | Easy | [`@dataclass`](../learning/python-reference-07-classes-and-dataclasses.ipynb#dataclass) |
| 7 | [Validate in __post_init__](#exercise-7-validate-in-__post_init__) | Medium | [`@dataclass`](../learning/python-reference-07-classes-and-dataclasses.ipynb#dataclass) |
| 8 | [Sortable records with order=True](#exercise-8-sortable-records-with-ordertrue) | Easy | [`@dataclass`](../learning/python-reference-07-classes-and-dataclasses.ipynb#dataclass) |
| 9 | [Parse statuses with an enum](#exercise-9-parse-statuses-with-an-enum) | Easy | [`enum`](../learning/python-reference-07-classes-and-dataclasses.ipynb#enum) |
| 10 | [Abstract sink with a template method](#exercise-10-abstract-sink-with-a-template-method) | Medium | [Abstract base classes and `Protocol`](../learning/python-reference-07-classes-and-dataclasses.ipynb#abstract-base-classes-and-protocol) |
| 11 | [Duck typing with a Protocol](#exercise-11-duck-typing-with-a-protocol) | Easy | [Abstract base classes and `Protocol`](../learning/python-reference-07-classes-and-dataclasses.ipynb#abstract-base-classes-and-protocol) |
| 12 | [Slots catch typos](#exercise-12-slots-catch-typos) | Easy | [`__slots__`](../learning/python-reference-07-classes-and-dataclasses.ipynb#__slots__) |
| 13 | [Cooperative mixin with super()](#exercise-13-cooperative-mixin-with-super) | Hard | [Inheritance, `super()` and the MRO](../learning/python-reference-07-classes-and-dataclasses.ipynb#inheritance-super-and-the-mro) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction

## Exercise 1: Alternate constructor with classmethod

**Difficulty:** Easy · about 5 min · **Practises:** [Attributes and methods](../learning/python-reference-07-classes-and-dataclasses.ipynb#attributes-and-methods)

*Building objects from config dicts or rows is the classic `classmethod` use.*

Write class `Connection(host, port=5432)` with `classmethod from_url(url)` parsing `host:port` (port optional) and a method `address()` returning `host:port`. Subclasses calling `from_url` must get their own type.

**Sample calls**

| call | expected |
| --- | --- |
| `Connection("db", 6543).address()` | `'db:6543'` |
| `Connection.from_url("db.internal:5433").address()` | `'db.internal:5433'` |
| `Connection.from_url("db").address()` | `'db:5432'` |

In [ ]:
class Connection:
    # your code here
    pass

In [ ]:
class Connection:
    def __init__(self, host, port=5432):
        self.host = host
        self.port = port

    @classmethod
    def from_url(cls, url):
        host, _, port = url.partition(":")
        return cls(host, int(port)) if port else cls(host)

    def address(self):
        return f"{self.host}:{self.port}"

## Exercise 2: Validated property

**Difficulty:** Easy · about 5 min · **Practises:** [Properties](../learning/python-reference-07-classes-and-dataclasses.ipynb#properties)

*Guarding invariants at assignment time instead of deep inside a pipeline.*

Write class `Threshold(value)` whose `value` property must be a number between 0 and 1 inclusive (raise `ValueError` otherwise, also in `__init__`) and a read-only property `percent` returning `value * 100` rounded to 1.

**Sample calls**

| call | expected |
| --- | --- |
| `Threshold(0.125).percent` | `12.5` |
| `(lambda t: (setattr(t, 'value', 1), t.percent)[1])(Threshold(0))` | `100` |
| `(Threshold(0).value, Threshold(1).value)` | `(0, 1)` |

In [ ]:
class Threshold:
    # your code here
    pass

In [ ]:
class Threshold:
    def __init__(self, value):
        self.value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, v):
        if isinstance(v, bool) or not isinstance(v, (int, float)) or not 0 <= v <= 1:
            raise ValueError(f"threshold must be within [0, 1], got {v!r}")
        self._value = v

    @property
    def percent(self):
        return round(self.value * 100, 1)

## Exercise 3: Money value object

**Difficulty:** Medium · about 15 min · **Practises:** [Representation, equality, hashing and ordering](../learning/python-reference-07-classes-and-dataclasses.ipynb#representation-equality-hashing-and-ordering)

*Value objects with correct equality and hashing are a common OOP interview task.*

Write class `Money(amount, currency)` storing `amount` as `Decimal`. Implement `__repr__` → `Money('12.50', 'USD')`, `__eq__` and `__hash__` on (amount, currency), `__add__` (same currency only, else `ValueError`; adding a non-Money returns `NotImplemented`), and `__radd__` so `sum()` starting from 0 works.

**Sample calls**

| call | expected |
| --- | --- |
| `repr(Money("12.50", "USD"))` | `"Money('12.50', 'USD')"` |
| `repr(Money("0.1", "USD") + Money("0.2", "USD"))` | `"Money('0.3', 'USD')"` |
| `repr(sum([Money(1, "EUR"), Money("2.5", "EUR")]))` | `"Money('3.5', 'EUR')"` |

In [ ]:
class Money:
    # your code here
    pass

In [ ]:
from decimal import Decimal

class Money:
    def __init__(self, amount, currency):
        self.amount = Decimal(str(amount))
        self.currency = currency

    def __repr__(self):
        return f"Money({str(self.amount)!r}, {self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __hash__(self):
        return hash((self.amount, self.currency))

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if other.currency != self.currency:
            raise ValueError(f"cannot add {self.currency} and {other.currency}")
        return Money(self.amount + other.amount, self.currency)

    def __radd__(self, other):
        if other == 0:
            return self
        return self.__add__(other)

## Exercise 4: Iterable container class

**Difficulty:** Medium · about 10 min · **Practises:** [Container and callable protocols](../learning/python-reference-07-classes-and-dataclasses.ipynb#container-and-callable-protocols)

*Custom collections should behave like built-ins.*

Write class `Partition(rows)` supporting `len()`, iteration, `in`, indexing, slicing (returns a `Partition`), and `bool()` false when empty. Add `column(name)` returning the list of that key's values.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda p: (len(p), list(p)))(Partition([{"a": 1}, {"a": 2}]))` | `(2, [{'a': 1}, {'a': 2}])` |
| `{"a": 2} in Partition([{"a": 2}])` | `True` |
| `Partition([{"a": 1}, {"a": 2}])[-1]` | `{'a': 2}` |

In [ ]:
class Partition:
    # your code here
    pass

In [ ]:
class Partition:
    def __init__(self, rows):
        self._rows = list(rows)

    def __len__(self):
        return len(self._rows)

    def __iter__(self):
        return iter(self._rows)

    def __contains__(self, row):
        return row in self._rows

    def __getitem__(self, index):
        result = self._rows[index]
        return Partition(result) if isinstance(index, slice) else result

    def column(self, name):
        return [row[name] for row in self._rows]

## Exercise 5: Comparable versions

**Difficulty:** Medium · about 10 min · **Practises:** [Representation, equality, hashing and ordering](../learning/python-reference-07-classes-and-dataclasses.ipynb#representation-equality-hashing-and-ordering)

*Sorting semantic versions correctly (1.10 > 1.9) is a real deployment-tooling task.*

Write class `SemVer(text)` parsing `major.minor.patch` (missing parts are 0) with `total_ordering`, `__eq__`, `__hash__`, `__lt__`, and `__str__` returning the normalised three-part form.

**Sample calls**

| call | expected |
| --- | --- |
| `[str(v) for v in sorted(map(SemVer, ["1.10.0", "1.9.3", "1.9", "0.1.2"]))]` | `['0.1.2', '1.9.0', '1.9.3', '1.10.0']` |
| `str(max(SemVer("2"), SemVer("1.99.99")))` | `'2.0.0'` |
| `SemVer("1.2") >= SemVer("1.2.0")` | `True` |

In [ ]:
class SemVer:
    # your code here
    pass

In [ ]:
from functools import total_ordering

@total_ordering
class SemVer:
    def __init__(self, text):
        parts = [int(p) for p in text.split(".")]
        self.parts = tuple(parts + [0] * (3 - len(parts)))

    def __eq__(self, other):
        if not isinstance(other, SemVer):
            return NotImplemented
        return self.parts == other.parts

    def __hash__(self):
        return hash(self.parts)

    def __lt__(self, other):
        if not isinstance(other, SemVer):
            return NotImplemented
        return self.parts < other.parts

    def __str__(self):
        return ".".join(map(str, self.parts))

## Exercise 6: Frozen dataclass for dedupe

**Difficulty:** Easy · about 5 min · **Practises:** [`@dataclass`](../learning/python-reference-07-classes-and-dataclasses.ipynb#dataclass)

*Hashable records let you deduplicate with a set.*

Write a frozen dataclass `Event(user_id: int, kind: str, ts: int)` and a function `dedupe(events)` returning the unique events in first-seen order, as tuples via `astuple`.

**Sample calls**

| call | expected |
| --- | --- |
| `dedupe([Event(1, "click", 5), Event(1, "click", 5), Event(2, "view", 5)])` | `[(1, 'click', 5), (2, 'view', 5)]` |
| `dedupe([])` | `[]` |
| `dedupe([Event(2, "a", 1), Event(1, "a", 1)])` | `[(2, 'a', 1), (1, 'a', 1)]` |

In [ ]:
def dedupe(events):
    # your code here
    pass

In [ ]:
from dataclasses import FrozenInstanceError, astuple, dataclass

@dataclass(frozen=True)
class Event:
    user_id: int
    kind: str
    ts: int

def dedupe(events):
    seen, out = set(), []
    for e in events:
        if e not in seen:
            seen.add(e)
            out.append(astuple(e))
    return out

## Exercise 7: Validate in __post_init__

**Difficulty:** Medium · about 10 min · **Practises:** [`@dataclass`](../learning/python-reference-07-classes-and-dataclasses.ipynb#dataclass)

*Records that refuse bad data at construction stop errors from spreading downstream.*

Write a dataclass `Job(name: str, schedule: str, retries: int = 3, tags: list[str] = <fresh list>)` whose `__post_init__` strips and lower-cases `name`, raises `ValueError` if the name is empty or `retries` is outside 0..10, and de-duplicates `tags` keeping order. Checks read fields with `asdict`.

**Sample calls**

| call | expected |
| --- | --- |
| `vars(Job("  Daily_Load ", "@daily", tags=["a", "b", "a"]))` | `{'name': 'daily_load', 'schedule': '@daily', 'retries': 3, 'tags': ['a', 'b']}` |
| `vars(Job("x", "@hourly"))` | `{'name': 'x', 'schedule': '@hourly', 'retries': 3, 'tags': []}` |
| `Job("a", "s").tags is not Job("b", "s").tags` | `True` |

In [ ]:
class Job:
    # your code here
    pass

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Job:
    name: str
    schedule: str
    retries: int = 3
    tags: list[str] = field(default_factory=list)

    def __post_init__(self):
        self.name = self.name.strip().lower()
        if not self.name:
            raise ValueError("name is required")
        if not 0 <= self.retries <= 10:
            raise ValueError(f"retries must be 0..10, got {self.retries}")
        self.tags = list(dict.fromkeys(self.tags))

## Exercise 8: Sortable records with order=True

**Difficulty:** Easy · about 5 min · **Practises:** [`@dataclass`](../learning/python-reference-07-classes-and-dataclasses.ipynb#dataclass)

*Ordering by priority then time for a work queue.*

Write a dataclass `Task(priority: int, created: int, name: str)` with `order=True`, where `name` does not take part in comparisons (`field(compare=False)`), and `queue_order(tasks)` returning names in sorted order (stable for equal keys).

**Sample calls**

| call | expected |
| --- | --- |
| `queue_order([Task(2, 1, "b"), Task(1, 5, "a")])` | `['a', 'b']` |
| `queue_order([Task(1, 9, "late"), Task(1, 2, "early")])` | `['early', 'late']` |
| `queue_order([Task(1, 1, "z"), Task(1, 1, "a")])` | `['z', 'a']` |

In [ ]:
def queue_order(tasks):
    # your code here
    pass

In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Task:
    priority: int
    created: int
    name: str = field(compare=False)

def queue_order(tasks):
    return [t.name for t in sorted(tasks)]

## Exercise 9: Parse statuses with an enum

**Difficulty:** Easy · about 5 min · **Practises:** [`enum`](../learning/python-reference-07-classes-and-dataclasses.ipynb#enum)

*Closed vocabularies (order status, file format) as enums catch typos early.*

Write `StrEnum` `Status` with members PLACED, PAID, SHIPPED, CANCELLED (values are lower-case names) and `parse_status(text)` accepting any case and surrounding spaces, returning the member's value; unknown → `ValueError`.

**Sample calls**

| call | expected |
| --- | --- |
| `parse_status("paid")` | `'paid'` |
| `parse_status("  Shipped ")` | `'shipped'` |
| `[parse_status(s) for s in ["PLACED", "cancelled"]]` | `['placed', 'cancelled']` |

In [ ]:
def parse_status(text):
    # your code here
    pass

In [ ]:
from enum import StrEnum, auto

class Status(StrEnum):
    PLACED = auto()
    PAID = auto()
    SHIPPED = auto()
    CANCELLED = auto()

def parse_status(text):
    return Status(text.strip().lower()).value

## Exercise 10: Abstract sink with a template method

**Difficulty:** Medium · about 10 min · **Practises:** [Abstract base classes and `Protocol`](../learning/python-reference-07-classes-and-dataclasses.ipynb#abstract-base-classes-and-protocol)

*Base classes with one abstract hook keep writers consistent.*

Write ABC `Sink` with abstract `write_batch(rows)` returning rows written, and a concrete `write(rows, size)` that splits rows into batches of `size` and returns the total. Then `ListSink(Sink)` storing batches in `self.batches`. Instantiating `Sink` directly must raise `TypeError`.

**Sample calls**

| call | expected |
| --- | --- |
| `ListSink().write(range(5), 2)` | `5` |
| `(lambda s: (s.write([1, 2, 3], 2), s.batches)[1])(ListSink())` | `[[1, 2], [3]]` |
| `ListSink().write([], 3)` | `0` |

In [ ]:
class Sink:
    # your code here
    pass


class ListSink:
    # your code here
    pass

In [ ]:
from abc import ABC, abstractmethod

class Sink(ABC):
    @abstractmethod
    def write_batch(self, rows):
        ...

    def write(self, rows, size):
        rows = list(rows)
        return sum(self.write_batch(rows[i:i + size]) for i in range(0, len(rows), size))

class ListSink(Sink):
    def __init__(self):
        self.batches = []

    def write_batch(self, rows):
        self.batches.append(rows)
        return len(rows)

## Exercise 11: Duck typing with a Protocol

**Difficulty:** Easy · about 5 min · **Practises:** [Abstract base classes and `Protocol`](../learning/python-reference-07-classes-and-dataclasses.ipynb#abstract-base-classes-and-protocol)

*Accepting anything with the right method, such as files, buffers or test doubles.*

Write `@runtime_checkable` Protocol `Readable` with `read(self, size: int = -1) -> str`, and `read_all(sources)` returning the concatenated text of sources that are `Readable`, skipping the rest.

**Sample calls**

| call | expected |
| --- | --- |
| `read_all([io.StringIO("ab"), io.StringIO("c")])` | `'abc'` |
| `read_all([42, "text", io.StringIO("x")])` | `'x'` |
| `read_all([])` | `''` |

In [ ]:
def read_all(sources):
    # your code here
    pass

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Readable(Protocol):
    def read(self, size: int = -1) -> str: ...

def read_all(sources):
    return "".join(s.read() for s in sources if isinstance(s, Readable))

## Exercise 12: Slots catch typos

**Difficulty:** Easy · about 5 min · **Practises:** [`__slots__`](../learning/python-reference-07-classes-and-dataclasses.ipynb#__slots__)

*Memory-light records for millions of rows, which also reject unknown attributes.*

Write class `Point(x, y)` with `__slots__` and a method `moved(dx, dy)` returning a new `Point`. Setting an unknown attribute must raise `AttributeError`.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda p: (p.x, p.y))(Point(1, 2).moved(3, -1))` | `(4, 1)` |
| `(lambda p: (p.moved(1, 1), (p.x, p.y))[1])(Point(0, 0))` | `(0, 0)` |
| `setattr(Point(0, 0), 'z', 1)` | raises `AttributeError` |

In [ ]:
class Point:
    # your code here
    pass

In [ ]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def moved(self, dx, dy):
        return Point(self.x + dx, self.y + dy)

## Exercise 13: Cooperative mixin with super()

**Difficulty:** Hard · about 15 min · **Practises:** [Inheritance, `super()` and the MRO](../learning/python-reference-07-classes-and-dataclasses.ipynb#inheritance-super-and-the-mro)

*Stacking behaviours (validation, logging) on readers with cooperative `super()` calls.*

Write `BaseReader(rows)` with `read()` returning a list copy of rows; `StripMixin` whose `read()` strips string values in each dict; `DropEmptyMixin` whose `read()` drops rows where every value is empty or None; and `CleanReader(DropEmptyMixin, StripMixin, BaseReader)`. Each mixin calls `super().read()`.

**Sample calls**

| call | expected |
| --- | --- |
| `CleanReader([{"a": " x ", "b": 1}, {"a": "  ", "b": None}]).read()` | `[{'a': 'x', 'b': 1}]` |
| `[c.__name__ for c in CleanReader.__mro__[:4]]` | `['CleanReader', 'DropEmptyMixin', 'StripMixin', 'BaseReader']` |
| `(lambda rows: (CleanReader(rows).read(), rows)[1])([{"a": " y "}])` | `[{'a': ' y '}]` |

In [ ]:
class BaseReader:
    pass


class StripMixin:
    pass


class DropEmptyMixin:
    pass


class CleanReader:
    # your code here
    pass

In [ ]:
class BaseReader:
    def __init__(self, rows):
        self.rows = rows

    def read(self):
        return [dict(r) for r in self.rows]

class StripMixin:
    def read(self):
        return [{k: v.strip() if isinstance(v, str) else v for k, v in r.items()} for r in super().read()]

class DropEmptyMixin:
    def read(self):
        return [r for r in super().read() if any(v not in ("", None) for v in r.values())]

class CleanReader(DropEmptyMixin, StripMixin, BaseReader):
    pass

**Next:** look anything up in [Python reference 7: classes, dataclasses and protocols](../learning/python-reference-07-classes-and-dataclasses.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).